# 🏥 介護施設統合AIシステム「ケア・リンク (Care-Link)」要約ノート

本ノートブックは、要求定義書・要件仕様書 (`system_requirements_specification.md`) および総合操作マニュアル (`USER_MANUAL.md`) を元に作成された **ケア・リンク (Care-Link)** システムのアーキテクチャ・機能構成・セキュリティ設計・開発ロードマップの統合要約レポートです。

---

## 📌 1. システム開発の背景と目的

日本の介護現場における要介護者の増加と人手不足を背景に、**自然言語AI・二重会話型AI (Full-Duplex AI)**、**完全ローカルLLM (Ollama/Gemma)**、**音声入出力 (Whisper/gTTS)** を統合し、以下の目的を達成します。

1. **自然な音声会話による傾聴と安心感の提供**: 超低遅延な会話型AIで利用者の不穏や孤立感を軽減。
2. **会話からの感情トーン把握**: 機嫌・不安・不穏を自動抽出し、スタッフの早期ケアに活用。
3. **自動バイタルサイン記録**: 発話（「熱は36度5分」）から自動解析しデータベースに保存。
4. **思い出のマルチメディア変換**: 対話内容をデジタル絵手紙・BGM・ショート動画へ自動生成しご家族と共有。
5. **完全オンプレミス処理による高度な個人情報保護**: 個人情報や医療記録は施設内Linuxサーバーで完結。


## 🗂️ 2. クライアント端末と 4つの動作モードコンセプト

ログイン時に `user_code` を入力すると、データベース内の権限に基づき以下の4つのモードへ動的に表示・機能を切り替えます。

| モード | 対象 | 主な機能とUI特徴 |
|---|---|---|
| **① 利用者モード (`patient`)** | 要介護者・入居者 | 32px以上の大文字高コントラストUI、1タップ音声対話、バイタル自動解析、インターホン受電 |
| **② スタッフモード (`staff`)** | 介護士・看護師・管理者 | 統合ダッシュボード、赤フラッシュアラート、リアルタイム会話監視・音声割り込み送信、インターホン発信、プロンプト雛形設定、申し送り・チャット |
| **③ ご家族モード (`family`)** | 利用者のご家族 | グループ境界アクセス制御、バイタル閲覧、対話サマリー・デジタル絵手紙/動画受領、面会予約 |
| **④ 訪問理美容モード (`barber`)** | 訪問理美容師 | 施術予約一覧、姿勢・認知症注意点（首傾斜不可等）の確認、施術完了報告入力 |


In [ ]:
# 4つの動作モードのロール定義とアクセス権限マトリクス
import pandas as pd

roles_data = {
    'Role': ['patient', 'staff', 'family', 'barber'],
    'User Target': ['利用者様', '介護スタッフ', 'ご家族様', '訪問理美容師'],
    'UI Contrast': ['極高(32px大文字)', '標準ダークモード', '標準モバイルレスポンシブ', '標準タスク指向'],
    'Access Scope': ['本人端末限定', '全施設・全利用者', 'グループ限定 (group_idフィルター)', '予約患者限定'],
    'Voice STT/TTS': ['有効 (Whisper/gTTS)', '有効 (割り込み・通話)', '閲覧のみ', '報告入力のみ']
}

df_roles = pd.DataFrame(roles_data)
df_roles


## 🔒 3. システムアーキテクチャ & セキュリティ設計

```text
┌─────────────────────────────────────────────────────────────────┐
│                    【クライアント端末群 (Webブラウザ)】          │
│   ① 利用者モード   ② スタッフモード   ③ ご家族モード   ④ 理美容  │
└────────────────────────────────┬────────────────────────────────┘
                                 │ (施設内LAN / WebSocket / HTTP REST)
                                 ▼
┌─────────────────────────────────────────────────────────────────┐
│     【施設内 Linux (Ubuntu) サーバーPC - 施設内閉域AI処理基盤】 │
│  ┌───────────────────────────────────────────────────────────┐  │
│  │  FastAPI (Python 3.12) - バックエンド API / WebSocket     │  │
│  ├───────────────┬───────────────┬───────────────┬───────────┤  │
│  │  音声処理基盤 │ 🔒 ローカルLLM│ 🗄️ SQLite 3   │ 🧠 RAG    │  │
│  │  (Whisper/gTTS)│ (Ollama Gemma)│ (Fernet暗号化)│ (ベクトル) │  │
│  └───────────────┴───────────────┴───────────────┴───────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

### 🔐 プライバシーと匿名化規定
- **ローカルAI処理**: 本人確認、バイタル解析、医療記録、申し送りは100%施設内LinuxサーバーのローカルLLM (Ollama / Gemma) で完結。
- **クラウドGemini-Live対話時の匿名化**: 一般会話（傾聴）のみクラウドAIを利用。実名は完全遮断し、**端末判定ニックネーム（例: なっちゃん）** のみ送信。
- **DB暗号化**: 氏名、発話ログ、特記事項は **Fernet (AES-128)** で暗号化保存。


In [ ]:
# バイタルサイン異常判定アルゴリズムの検証コード例
def validate_vitals(temp, bp_sys, bp_dia, weight):
    alerts = []
    if temp and temp >= 37.5:
        alerts.append(f'発熱警戒 (体温 {temp} ℃ >= 37.5℃)')
    if bp_sys and bp_sys >= 140:
        alerts.append(f'高血圧警戒 (収縮期血圧 {bp_sys} mmHg >= 140)')
    if bp_dia and bp_dia >= 90:
        alerts.append(f'拡張期高血圧 (拡張期血圧 {bp_dia} mmHg >= 90)')
    
    is_alert = len(alerts) > 0
    reason = ' / '.join(alerts) if is_alert else '正常'
    return is_alert, reason

# テスト実行
print('サンプル1:', validate_vitals(36.5, 120, 80, 60.0))
print('サンプル2 (高熱アラート):', validate_vitals(38.2, 125, 82, 60.0))
print('サンプル3 (高血圧アラート):', validate_vitals(36.8, 145, 95, 60.0))


## 💡 4. プロンプト雛形ライブラリ (認知症ペルソナ)

現場スタッフがワンタップで選択・適用できる 4つの標準プロンプトテンプレート：

1. 🌸 **受容・傾聴テンプレート (認知症・不穏対応)**:
   - 否定や訂正を一切せず、すべて「そうなんですね」「お気持ち分かりますよ」と肯定的に受容。
2. 📻 **回想療法・昔話テンプレート (昭和レトロ)**:
   - 「昔はどんなお仕事をされていたのですか？」と嬉しそうに語れる思い出を優しく引き出す。
3. ☀️ **意欲向上・アクティビティテンプレート (運動・散歩案内)**:
   - 「今日はお天気が良いので少しお庭を歩きませんか？」と散歩や水分補給を前向きに促す。
4. 🌇 **夕暮れ症候群・帰宅願望対応テンプレート (不安軽減)**:
   - 「家に帰りたい」という訴えに「帰れません」と否定せず、気持ちを受け止めて落ち着かせる。
